# **Import Library Example**

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import avg, col
from pyspark.sql.functions import month, year, sum as sum_
from pyspark.sql.functions import sum as sum_
from pyspark.sql.functions import col, sum as sum_, count
import matplotlib.pyplot as plt
from pyspark.sql.functions import sum
import pandas as pd
from pyspark.sql.functions import when
from pyspark.sql.functions import lit
import numpy as np
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, LogisticRegression
from pyspark.ml.regression import LinearRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.evaluation import ClusteringEvaluator
import seaborn as sns
import mlflow

# Data Loading and Import

In this step, the dataset is loaded into the Databricks environment using Apache Spark. The dataset is read from the specified file path in CSV format, with headers enabled and schema automatically inferred.

This process converts the raw dataset into a Spark DataFrame, which allows efficient distributed data processing and analysis.

In [0]:
df = spark.read.csv("/Workspace/Users/sinethmikariyawasam@gmail.com/Dataset/online_retail_II.csv", header=True, inferSchema=True)

# Dataset Overview

After loading the dataset, an initial exploration is performed to understand its structure, size, and attributes. This step is important to gain a clear understanding of the data before proceeding with preprocessing and analysis.

In [0]:
print("Row count:", df.count())
print("Column count:", len(df.columns))
print("Columns:", df.columns)

Row count: 1067371
Column count: 8
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [0]:
display(df.limit(10))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom


# Data Processing 

## 	Data cleaning

In [0]:
# Data Cleaning - Remove rows with nulls in key columns and filter out negative quantities
df_clean = df.dropna(subset=["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID"])
df_clean = df_clean.filter((df_clean["Quantity"] > 0) & (df_clean["Price"] > 0))

## Add a 'Sales' column

In [0]:
# Add a 'Sales' column
df_clean = df_clean.withColumn("Sales", col("Quantity") * col("Price"))

display(df_clean.limit(10))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom,83.4
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,81.0
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,81.0
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom,100.80000000000001
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,30.0
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01T07:45:00.000Z,1.65,13085.0,United Kingdom,39.599999999999994
489434,21871,SAVE THE PLANET MUG,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,30.0
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01T07:45:00.000Z,5.95,13085.0,United Kingdom,59.5
489435,22350,CAT BOWL,12,2009-12-01T07:46:00.000Z,2.55,13085.0,United Kingdom,30.599999999999998
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01T07:46:00.000Z,3.75,13085.0,United Kingdom,45.0


## Average sales per product

In [0]:
# Average sales per product
df_clean = df_clean.withColumnRenamed("Invoice", "InvoiceNo") \
                   .withColumnRenamed("Price", "UnitPrice") \
                   .withColumnRenamed("Customer ID", "CustomerID")

avg_sales_per_product = df_clean.groupBy("StockCode", "Description").agg(avg("Sales").alias("AvgSales")).orderBy("AvgSales", ascending=False)

In [0]:
display(avg_sales_per_product.limit(10))

StockCode,Description,AvgSales
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.6
22502,PICNIC BASKET WICKER 60 PIECES,19809.75
22790,"MIRROR, ARCHED GEORGIAN",3884.0000000000005
15059A,ENGLISH ROSE EDWARDIAN PARASOL,1515.0
85220,SMALL FAIRY CAKE FRIDGE MAGNETS,1375.0
23131,MISELTOE HEART WREATH CREAM,996.0000000000001
84760L,LARGE HANGING GLASS+ZINC LANTERN,896.0
DOT,DOTCOM POSTAGE,744.1475
84964B,BLUE PAINTED KASHMIRI TABLE,495.0
22833,HALL CABINET WITH 3 DRAWERS,493.6323076923076


Databricks visualization. Run in Databricks to view.

## Most popular products

In [0]:
# Most popular products (by total quantity sold)
popular_products = df_clean.groupBy("StockCode", "Description").agg(sum_("Quantity").alias("TotalQuantity")).orderBy("TotalQuantity", ascending=False)

In [0]:
display(popular_products.limit(10))

StockCode,Description,TotalQuantity
84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,109169
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93640
23843,"PAPER CRAFT , LITTLE BIRDIE",80995
84879,ASSORTED COLOUR BIRD ORNAMENT,79913
23166,MEDIUM CERAMIC TOP STORAGE JAR,77916
85099B,JUMBO BAG RED RETROSPOT,75759
17003,BROCADE RING PURSE,71129
21977,PACK OF 60 PINK PAISLEY CAKE CASES,55270
84991,60 TEATIME FAIRY CAKE CASES,53495
21212,PACK OF 72 RETROSPOT CAKE CASES,46107


Databricks visualization. Run in Databricks to view.

## Customer purchase patterns

In [0]:
# Customer purchase patterns (total sales and number of purchases per customer)
customer_patterns = df_clean.groupBy("CustomerID").agg({"Sales": "sum", "InvoiceNo": "count"}).withColumnRenamed("sum(Sales)", "TotalSales").withColumnRenamed("count(InvoiceNo)", "NumPurchases")

In [0]:
display(customer_patterns.limit(10 ))

CustomerID,TotalSales,NumPurchases
13457.0,148.22,9
13416.0,940.8,26
14286.0,8479.76,499
17047.0,2289.23,142
14285.0,3284.42,61
16782.0,10438.410000000007,1883
17866.0,1134.9499999999998,26
13050.0,12344.529999999999,1009
15184.0,689.95,85
17776.0,338.56,84


## Revenue trends over time

In [0]:
# Revenue trends over time (monthly revenue)
df_clean = df_clean.withColumn("Month", month(df_clean["InvoiceDate"])).withColumn("Year", year(df_clean["InvoiceDate"]))
monthly_revenue = df_clean.groupBy("Year", "Month").agg(sum_("Sales").alias("MonthlyRevenue")).orderBy("Year", "Month")

In [0]:
display(monthly_revenue.limit(10))

Year,Month,MonthlyRevenue
2009,12,686654.1599999949
2010,1,557319.0620000134
2010,2,506371.06600001536
2010,3,699608.9910000181
2010,4,594609.1919999976
2010,5,599985.7900000075
2010,6,639066.5800000058
2010,7,591636.7400000121
2010,8,604242.6499999989
2010,9,831615.0009999905


Databricks visualization. Run in Databricks to view.

In [0]:
null_counts = df_clean.select([col(c).isNull().cast("int").alias(c) for c in df_clean.columns]) \
    .agg(*[sum(col(c)).alias(c) for c in df_clean.columns])

display(null_counts)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Month,Year
0,0,0,0,0,0,0,0,0,0,0
